# Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
print(torch.__version__)


torch.manual_seed(42)

2.11.0+cpu


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import sys

# Pega aquí la ruta que copiaste entre las comillas:
ruta_de_mi_carpeta = '/content/drive/MyDrive/Colab_My_Projects/Deep_NN_Tensors_PyTorch'

# Le decimos a Colab que se mueva a esa carpeta:
os.chdir(ruta_de_mi_carpeta)
sys.path.append(ruta_de_mi_carpeta)

In [ ]:
# Convierte el archivo .ipynb a un .py real limpio
!jupyter nbconvert --to python /content/drive/MyDrive/Colab_My_Projects/Deep_NN_Tensors_PyTorch/dnn_utils.ipynb --output dnn_utils.py

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab_My_Projects/Deep_NN_Tensors_PyTorch/dnn_utils.ipynb to python
[NbConvertApp] Writing 8224 bytes to /content/drive/MyDrive/Colab_My_Projects/Deep_NN_Tensors_PyTorch/dnn_utils.py


In [ ]:
# 3. Cargar tu módulo
import dnn_utils
import importlib

# 4. Si modificas dnn_utils.py, ejecuta esta línea para recargar los cambios:
importlib.reload(dnn_utils)

from dnn_utils import *

# Understanding Overflow and Underflow and Their Effects on the Cost Function


**Objective:** Explore the role of floating-point precision in neural network numerical failures.

## Overflow

**Overflow** occurs when a number exceeds the maximum representable value in the floating-point format. Then, the system represents the result as $inf$ ($\infty$) or $-inf$ $(-\infty)$, depending  on the number.

<br>


To see the effect of overflow on the Cost function, we perform an experiment. We evaluate the model with  different parameter scales in order to examine the numerical behavior of the network. The scale is applied to the initial weights and biases, and the resulting output activations and cost is monitored.

In [ ]:
# Define the input  tensor X and the output tensor Y for the model
X = torch.rand(3,3)
Y = torch.tensor([[1., 0., 1.]])

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")

X shape: torch.Size([3, 3])
Y shape: torch.Size([1, 3])


In [ ]:
# Initialize parameters with scale
def initialize_parameters_scaled(layers, scale):
    torch.manual_seed(42)

    weights = {}
    biases = {}

    L = len(layers)

    for l in range(1, L):
        weights[l] = scale * torch.randn(layers[l], layers[l-1])
        biases[l] = scale * torch.randn(layers[l], 1)

    return weights, biases

Now, we compute the cost for different scales. In the last layer of the model, Sigmoid function is considered.

In [ ]:
# Construct the model for several scales

scales = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

for scale in scales:

    weights, biases = initialize_parameters_scaled(
        [3, 4, 2, 1], scale
    )

    AL, caches = forward_propagation(
        X, weights, biases
    )

    J = cost(AL, Y)

    print(
        f"scale = {scale:6g} | "
        f"cost = {J.item():.6f} | "
        f"AL min = {AL.min().item():.17e} | "
        f"AL max = {AL.max().item():.17e} | "
        f"finite = {torch.isfinite(J).item()}"
    )

scale =  0.001 | cost = 0.693155 | AL min = 4.99987542629241943e-01 | AL max = 4.99987542629241943e-01 | finite = True
scale =   0.01 | cost = 0.693228 | AL min = 4.99878317117691040e-01 | AL max = 4.99878436326980591e-01 | finite = True
scale =    0.1 | cost = 0.693825 | AL min = 4.98848110437393188e-01 | AL max = 4.98917490243911743e-01 | finite = True
scale =      1 | cost = 0.806018 | AL min = 2.95013636350631714e-01 | AL max = 3.55817943811416626e-01 | finite = True
scale =     10 | cost = nan | AL min = 0.00000000000000000e+00 | AL max = 0.00000000000000000e+00 | finite = False
scale =    100 | cost = nan | AL min = 0.00000000000000000e+00 | AL max = 0.00000000000000000e+00 | finite = False


What we observe from these results is that for the first three parameter scales, the output of the Sigmoid function ($AL$) remains close to 0.5 (as expected when the output logits $Z^L$ are close to zero), and the cost function has finite value.   


As the  parameter scale increases, $AL$ decreases, if the parameters take large values, the magnitude of the weights and biases raises considerably producing output logits $Z$ with larger magnitudes and hence with null values of $AL$ leading to a non-finite cost (nan).

For example, we consider the $scale = 10$:



In [ ]:
weights, biases = initialize_parameters_scaled(
        [3, 4, 2, 1], 10
    )

AL, records = forward_propagation(
    X, weights, biases
)

ZL = records[-1][1]

print("ZL:")
print(ZL)

print("ZL min:", ZL.min().item())
print("ZL max:", ZL.max().item())

print("AL:")
print(AL)

print("Exact zeros in AL:", torch.any(AL == 0).item())
print("Exact ones in AL:", torch.any(AL == 1).item())

J = cost(AL, Y)


ZL:
tensor([[-664.5488, -834.7203, -667.3497]])
ZL min: -834.7202758789062
ZL max: -664.5487670898438
AL:
tensor([[0., 0., 0.]])
Exact zeros in AL: True
Exact ones in AL: False


With a scale of 10 the logits take the negative large values:

$$
Z^L =
\begin{pmatrix}
-664.55 & -834.72 & -667.35
\end{pmatrix}.
$$

The Sigmoid function  

$$
A^L = \sigma(Z^L) =  \frac{1}{1 + e^{-Z^L}}
$$
<br>
is then evaluated at these values producing:
<br>

$$
A^L =
\begin{pmatrix}
0 & 0 & 0
\end{pmatrix}.
$$

Why do we get these exact zero  values? This behavior  originates in the exponential term of the Sigmoid function and the numerical computations which are performed using finite-precision arithmetic.





To see that, consider the first value of the vector $Z^L$, that is $-664.55$. To examine the numerical behavior of the Sigmoid function explicitly, we use NumPy  to evaluate the exponential term directly, since PyTorch's optimized torch.sigmoid() implementation returns the saturated value without exposing the underlying overflow.

In [ ]:
# Computing the Sigmoid function for the first value of ZL
Z_float32 = ZL[0][0].numpy()
print(f"ZL =", Z_float32)
print(f"ZL type: {Z_float32.dtype}")
print("----------------------------------")
sigmoid = 1 / (1 + np.exp(-Z_float32))


ZL = -664.54877
ZL type: float32
----------------------------------


/tmp/ipykernel_6109/2446942436.py:6: RuntimeWarning: overflow encountered in exp
  sigmoid = 1 / (1 + np.exp(-Z_float32))


NumPy returns an overflow warning.

For $Z=-664$, the sigmoid function computation requires evaluating $e^{664}$ which is of the order of $10^{288}$. This value exceeds the maximum representable value in float32 which is of the order of $10^{38}$, resulting in an overflow.





In [ ]:

np.exp(664)


np.float64(2.3525344061226884e+288)

In [ ]:
print(f"Sigmoid:", sigmoid)

Sigmoid: 0.0





Therefore, the subsequent division produces  an exact zero.


$$
\begin{aligned}
Z &= -664.55 \\
& \\
e^{-Z} &= e^{664.55}  \rightarrow \infty\\
& \\
A &= \frac{1}{1+e^{-Z}} =  \frac{1}{1+e^{664.55}} = \frac{1}{1+\infty} = 0
\end{aligned}
$$
<br>

The same is true for the rest of the values of $Z^L$. The fact that $A^L$ takes exact zero values is a serious problem for the the cost funtion because of the logarithm function involved in its computation:
<br><br>
$$
\text{for}\,\, A_i=0  \rightarrow \log(A_i) = \log(0) =-\infty,
$$
<br>
 which can lead to NaN values in the computed cost:
 <br><br>
$$J =
-\frac{1}{m}
\sum_{i=1}^{m}
\left[
Y_i\log(A^{L}_i)
+
(1-Y_i)\log(1-A^{L}_i)
\right]
$$
<br>

<br>
What about considering float 64 instead to solve this problem?
The same calculation can be examined using float64.

In [ ]:
# Computing the Sigmoid function with numpy for the first value of ZL
Z_float64 = Z_float32.astype(np.float64)
print(f"ZL type: {Z_float64.dtype}")
print("----------------------------------")
sigmoid = 1 / (1 + np.exp(-Z_float64))
print(f"Sigmoid:,", sigmoid)

ZL type: float64
----------------------------------
Sigmoid:, 2.4554861975217644e-289


For the case  of float64, the result of the Sigmoid function is a very small number and we don't have overflow.
This is because with float64,  we can represent numbers of the order of $10^{308}$,  so can still be stored as a finite value $e^{664}$.

However, in general,  increasing the floating-point precision does not ensure the elimination of  the underlying numerical problem. Sufficiently large values of the magnitude of $Z$ can eventually cause underflow  even in float64.









## Underflow

There is another point to consider, this is underflow.
**Underflow** occurs when a number becomes smaller in magnitude than the minimum representable positive value in the floating-point format. In this case, the numerical computation may represent the result as zero.

To observe underflow we perform the following experiment. This time we consider the model without scales, after forward propagation, we obtain  $A^{L-1}$, the activation in the layer previous to the last one.


In [ ]:
weights, biases = initialize_parameters([3, 4, 2, 1])

for l in range(1, len(weights)+1):
    print(f"\nLayer {l}:")
    print(f"W[{l}] shape: {weights[l].shape}")
    print(f"b[{l}] shape: {biases[l].shape}")


Layer 1:
W[1] shape: torch.Size([4, 3])
b[1] shape: torch.Size([4, 1])

Layer 2:
W[2] shape: torch.Size([2, 4])
b[2] shape: torch.Size([2, 1])

Layer 3:
W[3] shape: torch.Size([1, 2])
b[3] shape: torch.Size([1, 1])


In [ ]:
AL, records = forward_propagation(X, weights, biases)

A_prev = records[-1][0][0]

print("A^(L-1):")
print(A_prev)


A^(L-1):
tensor([[0.5913, 0.0479, 0.7052],
        [3.5776, 3.5504, 3.8390]])


In [ ]:
print("A_prev shape:", A_prev.shape)

A_prev shape: torch.Size([2, 3])


Hereinafter, we forget about the last layer in the model and construct a new one in order to get large positive values of $Z^L$.

In [ ]:
# Constructing weight, bias  and Z for the last layer
W_test = torch.ones(1, A_prev.shape[0],dtype=torch.float32) * 1000

b_test = torch.tensor(
    [[1000.0]],
    dtype=torch.float32
)

ZL_test = torch.einsum('ij,ja->ia', W_test, A_prev) + b_test

print("ZL_test:")
print(ZL_test)

ZL_test:
tensor([[5168.9185, 4598.2881, 5544.1763]])


For the positive large values
$$
Z^L = [5168.92, 4598.29, 5544.18]
$$

we have that
$$
e^{-Z^L} = [0, 0, 0]
$$
and hence the Sigmoid activation function in the last layer has the values
$$
A =  [1, 1, 1]
$$



In [ ]:
# Computing the Sigmoid function for the first value of ZL with numpy
ZL_np = ZL_test.numpy()
exp_np = np.exp(-ZL_np)
print(f"exp(-ZL_np):", exp_np)


AL_test_np= 1 / (1 + exp_np)
print(f"AL_np:", AL_test_np)
#exp_term = torch.exp(-ZL_test)

#print("exp(-ZL_test):")
#print(exp_term)

exp(-ZL_np): [[0. 0. 0.]]
AL_np: [[1. 1. 1.]]



When a positive value is smaller than the smallest positive normal value representable in the floating-point format (of the order of $10^{-38}$), it may underflow to zero.

Then, the term $e^{-Z^L}$ for large positive values of $Z^L$ is represented as zero, causing the Sigmoid output to become exactly $A^L=1$.




What effect does it has on the cost function?

In [ ]:
AL_test = torch.tensor(AL_test_np, dtype=torch.float32)
J_test = cost(AL_test, Y)

print("Y:", Y)
print("J_test:", J_test)

Y: tensor([[1., 0., 1.]])
J_test: tensor(nan)


For $Y=[1,0,1]$, the cost function involves $\log(1-A^L)$, which becomes $\log(0)$. In floating-point notation, this is represented as $-\infty$, causing the computed cost to become `NaN`.

Why we get a nan result?
<br>
For
$$
J=-\left[y\log(A)+(1-y)\log(1-A)\right]
$$

If $y=1$ and $A=1$ we have:

$$
J=-\left[1\log(1)+(1-1)\log(1-1)\right]
$$

$$
=-\left[1\cdot0+0\cdot\log(0)\right]
$$

In numerical computation

$$
\log(0)=-\infty
$$

and therefore,

$$
0\cdot(-\infty)=\mathrm{NaN}.
$$
In the second case
$$
y=0,\qquad A=1
$$

gives

$$
J=-\log(1-A)
$$

and therefore

$$
J=-\log(0)=+\infty.
$$



This illustrates an important distinction between the mathematical formulation and its finite-precision implementation: an expression can be mathematically well-defined while its direct numerical evaluation becomes unstable.

Conclusion: Numerical stability requires controlling the magnitude of the intermediate values $Z^{[l]}$, which can be affected by the scale of the weights, biases, and activations.

Note: with ZL_test being large and positive, exp(-ZL_test) underflows silently toward zero — it won't trigger RuntimeWarning: overflow encountered in exp. That warning only fires when the argument to exp() is very positive (like your earlier cell, where Z_32 was negative and you computed exp(-Z_32) = exp(664)). Underflow to zero, by contrast, usually raises no warning at all — it just rounds silently to 0.0. Worth noting in the notebook as an interesting asymmetry: NumPy warns on overflow, but not always on underflow.